# 1_2 Minute Level: Compute H and Nu

本 notebook 使用分钟级数据（默认 10:00-11:00，可配置）构建日度代理方差，不使用 tick/MUZ。随后估计 Hurst 指数并通过二阶矩回归截距得到 `nu_sq`。

In [6]:
import os
import sys
import datetime
import numpy as np
import pandas as pd
from dotenv import load_dotenv
load_dotenv()

sys.path.append('../models')
from DataFetcher import DataFetcher
from HurstEstimator import HurstEstimator

In [7]:
# 参数区
symbol = '000300.XSHG'
start_date = '20230101'
end_date = '20251231'

# 默认沿用 tick 主干时间窗，可按需调整
start_time = datetime.time(hour=10, minute=0)
end_time = datetime.time(hour=11, minute=0)
frequency = '1m'

license = os.environ.get('RQDATAC_LICENSE')

data_fetcher = DataFetcher()
data_fetcher.init_connection(license)

✓ 米筐数据连接成功


In [8]:
# 计算分钟级日度代理方差 RV = sum(r_t^2)
df_variance = data_fetcher.process_minute_variance_period(
    symbol=symbol,
    start_date=start_date,
    end_date=end_date,
    start_time=start_time,
    end_time=end_time,
    frequency=frequency,
    price_col='close',
    use_log_return=True,
    min_obs=5,
    verbose=True,
)

display(df_variance.head())
display(df_variance.tail())

分钟代理方差计算进度: 1/727 - 20230103
分钟代理方差计算进度: 21/727 - 20230207
分钟代理方差计算进度: 41/727 - 20230307
分钟代理方差计算进度: 61/727 - 20230404
分钟代理方差计算进度: 81/727 - 20230508
分钟代理方差计算进度: 101/727 - 20230605
分钟代理方差计算进度: 121/727 - 20230705
分钟代理方差计算进度: 141/727 - 20230802
分钟代理方差计算进度: 161/727 - 20230830
分钟代理方差计算进度: 181/727 - 20230927
分钟代理方差计算进度: 201/727 - 20231102
分钟代理方差计算进度: 221/727 - 20231130
分钟代理方差计算进度: 241/727 - 20231228
分钟代理方差计算进度: 261/727 - 20240126
分钟代理方差计算进度: 281/727 - 20240304
分钟代理方差计算进度: 301/727 - 20240401
分钟代理方差计算进度: 321/727 - 20240506
分钟代理方差计算进度: 341/727 - 20240603
分钟代理方差计算进度: 361/727 - 20240702
分钟代理方差计算进度: 381/727 - 20240730
分钟代理方差计算进度: 401/727 - 20240827
分钟代理方差计算进度: 421/727 - 20240926
分钟代理方差计算进度: 441/727 - 20241031
分钟代理方差计算进度: 461/727 - 20241128
分钟代理方差计算进度: 481/727 - 20241226
分钟代理方差计算进度: 501/727 - 20250124
分钟代理方差计算进度: 521/727 - 20250303
分钟代理方差计算进度: 541/727 - 20250331
分钟代理方差计算进度: 561/727 - 20250429
分钟代理方差计算进度: 581/727 - 20250530
分钟代理方差计算进度: 601/727 - 20250630
分钟代理方差计算进度: 621/727 - 20250728
分钟代理方差计算进度: 64

,rv_minute,n_observations,n_returns,annualized_vol
date,,,,
2023-01-03,0.000011,61,60,5.301845
2023-01-04,0.000008,61,60,4.538374
2023-01-05,0.000011,61,60,5.196513
2023-01-06,0.000006,61,60,3.983375
2023-01-09,0.000010,61,60,4.978114


,rv_minute,n_observations,n_returns,annualized_vol
date,,,,
2025-12-25,0.000006,61,60,3.818633
2025-12-26,0.000009,61,60,4.746399
2025-12-29,0.000009,61,60,4.690114
2025-12-30,0.000010,61,60,4.974260
2025-12-31,0.000009,61,60,4.735590


In [9]:
# 估计 H 与 nu_sq
df_variance = df_variance[df_variance['rv_minute'] > 0].copy()
log_sigma_series = 0.5 * np.log(df_variance['rv_minute'])

hurst = HurstEstimator()
H, h_info = hurst.estimate_hurst_variogram(log_sigma_series, q=2.0)
nu_sq = float(np.exp(h_info['intercept']))

print(f'H = {H:.6f}')
print(f'R^2 = {h_info.get("r_squared", np.nan):.6f}')
print(f'nu_sq = {nu_sq:.10f}')

H = 0.122778
R^2 = 0.890981
nu_sq = 0.0879455542


In [10]:
# 保存分钟代理方差文件
output_dir = '../data'
os.makedirs(output_dir, exist_ok=True)

output_file = os.path.join(
    output_dir,
    f'variance_proxy_minute_level_{start_date}_{end_date}.csv'
)
df_variance.to_csv(output_file)
print(f'✓ 数据已保存至: {output_file}')

✓ 数据已保存至: ../data/variance_proxy_minute_level_20230101_20251231.csv
